In [1]:
import os 
from getpass import getpass

username = os.environ["Grazioso_User"] = getpass("DB username: ")
password = os.environ["Grazioso_Password"] = getpass("DB password: ")

In [ ]:
from dash import Dash # Setup should use Dash no Jupyterdash as this Dash version already supports the notebook

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html, ctx # Lets the dashboard figure out what sorting control the user just changed
from animal_merge_sort import merge_sort
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
# JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# Read the database login from enviornment
# Keeps sensitive information out of code
username = os.environ.get("Grazioso_User")
password = os.environ.get("Grazioso_Password")

# Stop with an informative message if needed info is missing
if not username or not password:
    raise RuntimeError("Set Grazioso_User and Grazioso_Password before running.")

# Connect to database via CRUD Module by passing in protected login
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
# error="ignore' prevents crash if there is no _id column
df.drop(columns=['_id'],inplace=True, errors="ignore")

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = Dash(__name__)

#Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#Place the HTML image tag in the line below into the app.layout code according to your design
#Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
    html.Div(id='hidden-div', style={'display':'none'}),
    dcc.Store(id="rescue-filter", data="all"), # Remembers rescue category when sorting controls change
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.Center(html.A(html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode())), href='https://www.snhu.edu', target='_blank')),
    html.Center(html.B(html.H2('By Nicholas Deniz'))),
    html.Hr(),
    html.Div(className='buttonRow',
             style={'display' : 'flex'},
                 children=[
                     html.Button(id='water-button', n_clicks=0, children='Water Rescue'), 
                     html.Button(id='mountain-button', n_clicks=0, children='Mountain or Wilderness Rescue'), 
                     html.Button(id='disaster-button', n_clicks=0, children='Disaster or Individual Tracking'), 
                     html.Button(id='reset-button', n_clicks=0, children='Reset'), 
                  ]
# Add in code for the interactive filtering options. For example, Radio buttons, drop down, checkboxes, etc.
    ),
    html.Hr(),
    # Lets user choose field and direction for the merge sort
    html.Div([
        html.Label("Sort animal records by"),
        dcc.Dropdown(
            id="sort-field",
            options=[
                {
                    "label": "Age in weeks",
                    "value": "age_upon_outcome_in_weeks",
                },
                {
                    "label": "Name",
                    "value": "name",
                },
                {
                    "label": "Breed",
                    "value": "breed",
                },
            ],
            # Start the sort by age
            value="age_upon_outcome_in_weeks",
            clearable=False,
        ),
        html.Label("Order"),

        dcc.RadioItems(
            id="sort-direction",
            options=[
                {
                    "label": "Ascending",
                    "value": "ascending",
                },
                {
                    "label": "Descending",
                    "value": "descending",
                },
            ],
            # Start with the smallest value first
            value="ascending",
            labelStyle={
                "display": "inline-block",
                "marginRight": "10px",
            },
        ),
    ], style={"maxWidth": "300px"}),

    html.Hr(),

#Set up the features for your interactive data table to make it user-friendly for your client
#If you completed the Module Six Assignment, you can copy in the code you created here 
dash_table.DataTable(
    id='datatable-id',
    columns=[
        {"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns
    ],
    # Use merge sort for the starting order
    data=merge_sort(
        df.to_dict("records"),
        "age_upon_outcome_in_weeks",
    ),
    #Set up the features for your interactive data table to make it user-friendly for your client
    row_selectable='single',
    column_selectable="single",
    selected_rows=[0] if not df.empty else [],
    sort_action='none', # Getting rid of Dash'sbuilt in sort
    filter_action='native',
    page_action='native',
    page_current= 0,
    page_size= 10,

    style_table={
        "overflowX": "auto",
    },

    style_header={
        "backgroundColor": "white",
        "color": "black",
    },

    style_data={
        "backgroundColor": "white",
        "color": "black",
    },
),   

html.Br(),
html.Hr(),
                                
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',)
            ]
        )
])

#############################################
# Interaction Between Components / Controller
#############################################

@app.callback([Output("datatable-id", "data"), Output("datatable-id", "selected_rows"), Output("datatable-id", "page_current"), Output("rescue-filter", "data"),], 
              [Input("water-button", "n_clicks"), Input("mountain-button", "n_clicks"), Input("disaster-button", "n_clicks"), Input("reset-button", "n_clicks"),Input("sort-field", "value"),Input("sort-direction", "value"),], [State("rescue-filter", "data"),],)

def update_dashboard(
    _water_clicks,
    _mountain_clicks,
    _disaster_clicks,
    _reset_clicks,
    sort_field,
    sort_direction,
    rescue_filter,
):
    """Get matching animals and sort the already loaded in records"""

    # Figure out which dashboard control was recently changed
    changed_id = ctx.triggered_id

    # Only a rescure-button click will changesaved rescue category
    # Changing the sorting controls will keep the current rescure category
    if changed_id == "water-button":
        rescue_filter = "water"

    elif changed_id == "mountain-button":
        rescue_filter = "mountain"

    elif changed_id == "disaster-button":
        rescue_filter = "disaster"

    elif changed_id == "reset-button":
        rescue_filter = "all"

    # Empty MongoDB query will return all animal records
    query = {}

#Water rescue
    if rescue_filter == "water":
        query = {
            "animal_type": "Dog", 
            "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
        }
        
    #Mountain or wilderness Resuce
    elif rescue_filter == "mountain":
        query = {
            "animal_type": "Dog", 
            "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
        }
        
    #Disaster or Individual Tracking Resuce
    elif rescue_filter == "disaster":
        query = {
            "animal_type": "Dog", 
            "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300},
        }

    # MongoDB gets animals matching the rescue category
    records = db.read(query)

    # Removes MongoDB's Object ID since Dash can't display it
    for record in records:
        record.pop("_id", None)

    # Sorts records returned by MongoDB
    sorted_records = merge_sort(
        records,
        sort_field,
        descending=(sort_direction == "descending"),
    )

    selected_rows = [0] if sorted_records else []
    # Display sorted records and remember rescue category. And resetting page_current with prevent the table for displaying as blank when fewer pages are shown than the previous page
    return sorted_records, selected_rows, 0, rescue_filter

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(Output('graph-id', "children"), [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    # add code for chart of your choice (e.g. pie chart) #
    if not viewData:
        return [html.P("There are no animals that match the fitlers")]
    
    graph_data = pd.DataFrame.from_records(viewData)

    return [dcc.Graph(figure = px.pie(graph_data, names='breed', title='Preferred Animals'))]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(Output('datatable-id', 'style_data_conditional'),[Input('datatable-id', 'selected_columns')])
def update_styles(selected_columns):
    #If nothing is selected return nothing
    if selected_columns is None:
        return[]
    
    return [{
        'if': { 'column_id': i },
        'backgroundColor': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(Output('map-id', "children"), [Input('datatable-id', "derived_virtual_data"), Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if not viewData or not index:
        return []
    
    dff = pd.DataFrame.from_dict(viewData)
    row = index[0]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, 
            children=[
                dl.TileLayer(id="base-layer-id"),
                # Marker with tool tip and popup
                # Column 13 and 14 define the grid-coordinates for the map
                # Column 4 defines the breed for the animal
                # Column 9 defines the name of the animal
                dl.Marker(
                    position=[dff.iloc[row,13],dff.iloc[row,14]], 
                    children=[
                        dl.Tooltip(dff.iloc[row,4]),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(dff.iloc[row,9])
                        ])
                    ]
                )
            ]
        )
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run(jupyter_mode='inline', debug=False) 

: 